# Chaîne de modélisation CNIG : UML → JSON Schema → OWL/RDF

Analyse de la chaîne de modélisation conceptuelle des standards CNIG et formalisation en ontologie OWL.  
Ce notebook est complémentaire à `notebook_cnig_rag.ipynb` — il documente la construction de `friches_ontologie.owl` et `friches_ontologie.ttl`.

```
UML → JSON Schema (Frictionless) → OWL/RDF → RAG
```

Références :
- Standard source : https://github.com/cnigfr/schema-friches  
- W3C OWL 2 : https://www.w3.org/TR/owl2-primer/  
- SEMIC GeoDCAT-AP : https://github.com/SEMICeu/GeoDCAT-AP

## 1. Analyse du JSON Schema — reconstruction de la structure UML

Le standard CNIG Friches est distribué en Frictionless Data Table Schema.  
La structure UML sous-jacente se lit dans les préfixes des noms de champs.

In [ ]:
import json
from collections import defaultdict

with open('schema_friches.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)

fields = schema.get('fields', [])
print(f"Standard : {schema.get('title')} v{schema.get('version')}")
print(f"Champs   : {len(fields)}")
print(f"Format   : Frictionless Data Table Schema\n")

# Reconstruction des classes UML depuis les préfixes
# Mapping préfixe → classe UML d'après le modèle conceptuel du standard
PREFIX_TO_CLASS = {
    'site':    'Friche',
    'comm':    'Localisation',
    'dep':     'Localisation',
    'reg':     'Localisation',
    'bati':    'Batiment',
    'sol':     'Pollution',
    'urba':    'Urbanisme',
    'proprio': 'Proprietaire',
    'source':  'Source',
    'acti':    'Activite'
}

classes = defaultdict(list)
for f in fields:
    prefix = f['name'].split('_')[0]
    classe = PREFIX_TO_CLASS.get(prefix, f'? ({prefix})')
    classes[classe].append(f['name'])

print(f"{'Classe UML':<18} {'Nb champs':<12} Exemples de champs")
print('-' * 65)
for classe, champs in sorted(classes.items()):
    print(f"{classe:<18} {len(champs):<12} {', '.join(champs[:3])}{'...' if len(champs) > 3 else ''}")

## 2. Construction de l'ontologie OWL

Formalisation du modèle conceptuel en OWL via owlready2.  
Les Object Properties modélisent les associations UML, les Data Properties les attributs.

In [ ]:
from owlready2 import *

onto = get_ontology("http://cnig.gouv.fr/ontologies/friches#")

with onto:

    # Classes — calées sur le modèle UML du standard
    class Friche(Thing): pass
    class Localisation(Thing): pass
    class Batiment(Thing): pass
    class Proprietaire(Thing): pass
    class Pollution(Thing): pass
    class Urbanisme(Thing): pass
    class Source(Thing): pass
    class Activite(Thing): pass

    # Object Properties — associations UML entre classes
    class aLocalisation(ObjectProperty):
        domain = [Friche]; range = [Localisation]

    class aBatiment(ObjectProperty):
        domain = [Friche]; range = [Batiment]

    class aProprietaire(ObjectProperty):
        domain = [Friche]; range = [Proprietaire]

    class aPollution(ObjectProperty):
        domain = [Friche]; range = [Pollution]

    class aUrbanisme(ObjectProperty):
        domain = [Friche]; range = [Urbanisme]

    class aSource(ObjectProperty):
        domain = [Friche]; range = [Source]

    class aActivite(ObjectProperty):
        domain = [Friche]; range = [Activite]

    # Data Properties — attributs, types calés sur Frictionless Data
    class site_id(DataProperty):              domain=[Friche];        range=[str]
    class site_nom(DataProperty):             domain=[Friche];        range=[str]
    class site_type(DataProperty):            domain=[Friche];        range=[str]
    class site_statut(DataProperty):          domain=[Friche];        range=[str]
    class site_surface(DataProperty):         domain=[Friche];        range=[float]
    class comm_nom(DataProperty):             domain=[Localisation];  range=[str]
    class comm_insee(DataProperty):           domain=[Localisation];  range=[str]
    class dep_nom(DataProperty):              domain=[Localisation];  range=[str]
    class reg_nom(DataProperty):              domain=[Localisation];  range=[str]
    class bati_nombre(DataProperty):          domain=[Batiment];      range=[int]
    class bati_surface(DataProperty):         domain=[Batiment];      range=[float]
    class bati_etat(DataProperty):            domain=[Batiment];      range=[str]
    class sol_pollution_existe(DataProperty): domain=[Pollution];     range=[str]
    class sol_pollution_origine(DataProperty):domain=[Pollution];     range=[str]
    class urba_zone_type(DataProperty):       domain=[Urbanisme];     range=[str]
    class urba_zone_lib(DataProperty):        domain=[Urbanisme];     range=[str]
    class proprio_type(DataProperty):         domain=[Proprietaire];  range=[str]
    class proprio_nom(DataProperty):          domain=[Proprietaire];  range=[str]
    class source_nom(DataProperty):           domain=[Source];        range=[str]
    class source_producteur(DataProperty):    domain=[Source];        range=[str]
    class acti_nom(DataProperty):             domain=[Activite];      range=[str]
    class acti_secteur(DataProperty):         domain=[Activite];      range=[str]

onto.save(file="friches_ontologie.owl", format="rdfxml")

print(f"✅ Ontologie construite et sauvegardée")
print(f"   Classes           : {len(list(onto.classes()))}")
print(f"   Object properties : {len(list(onto.object_properties()))}")
print(f"   Data properties   : {len(list(onto.data_properties()))}")

## 3. Introspection et génération du contexte pour le RAG

In [ ]:
# Vérification de la structure de l'ontologie
print(f"{'Object Property':<20} {'Domaine':<16} {'Portée'}")
print('-' * 50)
for p in onto.object_properties():
    d = p.domain[0].name if p.domain else '?'
    r = p.range[0].name  if p.range  else '?'
    print(f"{p.name:<20} {d:<16} {r}")

print()
print(f"{'Data Property':<32} {'Domaine':<16} {'Type'}")
print('-' * 65)
xsd_map = {'str': 'xsd:string', 'int': 'xsd:integer', 'float': 'xsd:float'}
for p in onto.data_properties():
    d = p.domain[0].name if p.domain else '?'
    t = xsd_map.get(p.range[0].__name__ if p.range else '?', '?')
    print(f"{p.name:<32} {d:<16} {t}")

In [ ]:
# Génération du contexte ontologique pour injection dans le system prompt RAG
# Utilisé dans notebook_cnig_rag.ipynb — Bloc 6

def build_ontology_context(onto):
    classes    = [c.name for c in onto.classes()]
    obj_props  = [f"{p.name}: {p.domain[0].name if p.domain else '?'} → {p.range[0].name if p.range else '?'}"
                  for p in onto.object_properties()]
    data_props = [f"{p.name} (domaine: {p.domain[0].name if p.domain else '?'})"
                  for p in onto.data_properties()]

    return f"""ONTOLOGIE CNIG FRICHES

Classes : {', '.join(classes)}

Relations entre classes :
{chr(10).join(['  - ' + r for r in obj_props])}

Propriétés des données :
{chr(10).join(['  - ' + d for d in data_props])}

Connexions inter-standards identifiées :
  - urba_zone_type (U/AU/A/N) → défini dans le standard CNIG PLU
  - acti_secteur → correspondance potentielle avec CNIG Sites économiques
  - sol_pollution_* → lien thématique avec CNIG Geostandards-Risques
"""

ctx = build_ontology_context(onto)
print(ctx)
print(f"Longueur du contexte : {len(ctx)} caractères")

## 4. Correspondances sémantiques inter-standards

Identification manuelle des connexions entre le standard Friches et les autres standards CNIG.  
Ces correspondances sont la base du livrable 5 (connexions sémantiques) — à automatiser via GraphRAG ou OG-RAG multi-corpus.

In [ ]:
import pandas as pd

# Correspondances identifiées manuellement — à valider avec les GT CNIG
correspondances = pd.DataFrame([
    {
        'champ_friches':   'urba_zone_type',
        'classe_friches':  'Urbanisme',
        'standard_cible':  'CNIG PLU',
        'concept_cible':   'typeZone',
        'type_lien':       'skos:exactMatch',
        'confiance':       'haute — valeurs identiques (U/AU/A/N)'
    },
    {
        'champ_friches':   'acti_secteur',
        'classe_friches':  'Activite',
        'standard_cible':  'CNIG Sites économiques',
        'concept_cible':   'secteur_activite',
        'type_lien':       'skos:relatedMatch',
        'confiance':       'moyenne — à valider GT'
    },
    {
        'champ_friches':   'sol_pollution_existe',
        'classe_friches':  'Pollution',
        'standard_cible':  'CNIG Geostandards-Risques',
        'concept_cible':   'pollution_sol',
        'type_lien':       'skos:relatedMatch',
        'confiance':       'à explorer — lien thématique fort'
    },
])

print("Correspondances sémantiques inter-standards CNIG")
print(correspondances.to_string(index=False))
print()
print("Automatisation possible via :")
print("  - GraphRAG (Microsoft) : détection automatique de correspondances entre corpus")
print("  - OG-RAG multi-corpus  : extension de ce projet à N standards simultanément")
print("  - LLM + prompting      : demander au LLM d'identifier les équivalences entre standards")